## NLP with Disaster Tweets

This project classifies tweets as being about a real disaster or not. Some tweets use disaster-related words metaphorically (e.g. "this song is fire"), which makes it a genuinely hard text classification problem, not just keyword matching.

## Approach
1. Load the data and explore the target balance and a few example tweets
2. Convert tweet text into numeric features with TF-IDF
3. Train a baseline classifier and evaluate F1 on a validation split
4. Generate predictions and create the submission file
5. Submit to Kaggle and record the score

In [2]:
import os
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("Paste your Kaggle API token and press Enter: ")

!pip install -q -U kaggle
!kaggle competitions download -c nlp-getting-started
!unzip -oq nlp-getting-started.zip

Paste your Kaggle API token and press Enter: ··········
100% 593k/593k [00:00<00:00, 852kB/s]



In [3]:
import pandas as pd

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [4]:
print(train.shape, test.shape)

(7613, 5) (3263, 4)


In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7613 entries, 0 to 7612
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        7613 non-null   int64 
 1   keyword   7552 non-null   object
 2   location  5080 non-null   object
 3   text      7613 non-null   object
 4   target    7613 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 297.5+ KB


In [6]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3263 entries, 0 to 3262
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        3263 non-null   int64 
 1   keyword   3237 non-null   object
 2   location  2158 non-null   object
 3   text      3263 non-null   object
dtypes: int64(1), object(3)
memory usage: 102.1+ KB


In [7]:
train.sample(1)

,id,keyword,location,text,target
5762,8224,riot,"Frankfort, KY",Transwomen and drag queens of color lead the r...,0


In [8]:
test.sample(1)

,id,keyword,location,text
1194,3929,devastated,NaN,Actually if I don't see Hunter Hayes and Lady ...


In [9]:
train["target"].value_counts(normalize=True)

,proportion
target,
0,0.57034
1,0.42966


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000, stop_words="english")
x = vectorizer.fit_transform(train["text"])
y = train["target"]

In [12]:
print(x.shape, y.shape)

(7613, 5000) (7613,)


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)

pred = model.predict(x_val)
print("Validation F1:", round(f1_score(y_val, pred), 4))

Validation F1: 0.7656


In [14]:
model.fit(x, y)

x_test = vectorizer.transform(test["text"])
pred_test = model.predict(x_test)

pd.DataFrame({"id": test["id"], "target": pred_test}).to_csv("submission.csv", index=False)

In [15]:
!kaggle competitions submit -c nlp-getting-started -f submission.csv -m "TF-IDF + LogisticRegression baseline"

100% 22.2k/22.2k [00:01<00:00, 17.4kB/s]
4 submissions remaining today.
Successfully submitted to Natural Language Processing with Disaster Tweets